## Text Mining Lab

Conor Murray 

In [1]:
import pandas as pd
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')      
nltk.download('stopwords')
nltk.download('vader_lexicon')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\conor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\conor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\conor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\conor\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [2]:
df = pd.read_csv('amazon_labelled.txt', sep='\t', quoting=3)
s = df.loc[26, 'text']
print(s)

I've owned this phone for 7 months now and can say that it's the best mobile phone I've had.


In [3]:
from nltk.tokenize import word_tokenize
sSplit = s.split()
sTokenized = word_tokenize(s)
print("split():")
print(sSplit)
print()
print("word_tokenize():")
print(sTokenized)

split():
["I've", 'owned', 'this', 'phone', 'for', '7', 'months', 'now', 'and', 'can', 'say', 'that', "it's", 'the', 'best', 'mobile', 'phone', "I've", 'had.']

word_tokenize():
['I', "'ve", 'owned', 'this', 'phone', 'for', '7', 'months', 'now', 'and', 'can', 'say', 'that', 'it', "'s", 'the', 'best', 'mobile', 'phone', 'I', "'ve", 'had', '.']


Quesiton 1: split() only breaks on white space not on punctuation, this means phone and phone. are characterized as two different words whereas word_tokenize serperates punctiation into its own token.

In [4]:
def fix_contraction(word):
    if word == "'ve":
        return "have"
    elif word == "'s":
        return "is"
    else:
        return word
newTokens = []
for word in sTokenized:
    newTokens.append(fix_contraction(word))
sTokenized = newTokens
newTokens = []
for word in sTokenized:
    if word.isalpha():
        newTokens.append(word)
sTokenized = newTokens
newTokens = []
for word in sTokenized:
    newTokens.append(word.lower())
sTokenized = newTokens
print(sTokenized)

['i', 'have', 'owned', 'this', 'phone', 'for', 'months', 'now', 'and', 'can', 'say', 'that', 'it', 'is', 'the', 'best', 'mobile', 'phone', 'i', 'have', 'had']


In [5]:
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
stemmer = PorterStemmer()
stopWords = stopwords.words('english')
newTokens = []
for word in sTokenized:
    if word not in stopWords:
        newTokens.append(stemmer.stem(word))
sTokenized = newTokens
print("Number of tokens remaining:", len(sTokenized))
print(sTokenized)

Number of tokens remaining: 7
['own', 'phone', 'month', 'say', 'best', 'mobil', 'phone']


Question 2: 7 tokens remain after performing these operations

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
analyzer = CountVectorizer().build_analyzer()
def stemmed_words(doc):
    result = []
    for word in analyzer(doc):
        if word not in stopWords and word.isalpha():
            result.append(stemmer.stem(word))
    return result
cv = CountVectorizer(analyzer=stemmed_words)
X = cv.fit_transform(df['text'])
dfX = pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out())
print(dfX.shape)
print(dfX.head())

(1000, 1352)
   abhor  abil  abl  abound  absolut  absolutel  ac  accept  access  \
0      0     0    0       0        0          0   0       0       0   
1      0     0    0       0        0          0   0       0       0   
2      0     0    0       0        0          0   0       0       0   
3      0     0    0       0        0          0   0       0       0   
4      0     0    0       0        0          0   0       0       0   

   accessori  ...  worthwhil  would  wow  wrong  wrongli  ye  year  yell  yet  \
0          0  ...          0      0    0      0        0   0     0     0    0   
1          0  ...          0      0    0      0        0   0     0     0    0   
2          0  ...          0      0    0      0        0   0     0     0    0   
3          0  ...          0      0    0      0        0   0     0     0    0   
4          0  ...          0      0    0      0        0   0     0     0    0   

   zero  
0     0  
1     0  
2     0  
3     0  
4     0  

[5 rows x 13

In [7]:
from sklearn.model_selection import train_test_split
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(dfX, y, test_size=0.3, random_state=1)
print(X_train.shape, X_test.shape)

(700, 1352) (300, 1352)


In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
dt = DecisionTreeClassifier()
dt = dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
nn = MLPClassifier()
nn = nn.fit(X_train, y_train)
nn_pred = nn.predict(X_test)
print("Question 3:")
print()
print("Decision Tree")
print("Accuracy: ", accuracy_score(y_test, dt_pred))
print("Precision:", precision_score(y_test, dt_pred))
print("Recall:   ", recall_score(y_test, dt_pred))
print()
print("Neural Network")
print("Accuracy: ", accuracy_score(y_test, nn_pred))
print("Precision:", precision_score(y_test, nn_pred))
print("Recall:   ", recall_score(y_test, nn_pred))

Question 3:

Decision Tree
Accuracy:  0.73
Precision: 0.75
Recall:    0.6503496503496503

Neural Network
Accuracy:  0.8
Precision: 0.8074074074074075
Recall:    0.7622377622377622


In [13]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sid = SentimentIntensityAnalyzer()
print(sid.polarity_scores(s))

{'neg': 0.0, 'neu': 0.802, 'pos': 0.198, 'compound': 0.6369}


In [15]:
def get_pos(text):
    scores = sid.polarity_scores(text)
    return scores['pos']
def get_neg(text):
    scores = sid.polarity_scores(text)
    return scores['neg']
posScores = []
negScores = []
for i in range(len(df)):
    posScores.append(get_pos(df.loc[i, 'text']))
    negScores.append(get_neg(df.loc[i, 'text']))
df['pos'] = posScores
df['neg'] = negScores
def compare_scores(pos, neg):
    if pos > neg:
        return 1
    else:
        return 0
vaderScores = []
for i in range(len(df)):
    vaderScores.append(compare_scores(df.loc[i, 'pos'], df.loc[i, 'neg']))
df['vader'] = vaderScores
print(df.head())

                                                text  sentiment    pos    neg  \
0  So there is no way for me to plug it in here i...          0  0.000  0.120   
1                        Good case, Excellent value.          1  0.900  0.000   
2                             Great for the jawbone.          1  0.577  0.000   
3  Tied to charger for conversations lasting more...          0  0.000  0.287   
4                                  The mic is great.          1  0.577  0.000   

   vader  
0      0  
1      1  
2      1  
3      0  
4      1  


In [12]:
mask = df['vader'] == df['sentiment']
matchPct = mask.sum() / len(df) * 100
print("Percentage of matches:", matchPct)

Percentage of matches: 83.2


Question 4: The VADER indicator had a 83.2% match rate which was better than the Nerutal Network at 80% and Decision Tree at 73%. The decision tree did worse since ti tends to overfit small data and the neural network scored slightly worse becuase it only used the 30% test set. this shows an advantage of VADER, it reads 100% of the original text and doesnt depend on training data so it resits overfitting and small data sets.